# A Surface Mass Balance (SMB) emulator over Antarctica


This notebook is a demonstrator from the **Digital Twin Component for Ice Sheets (DTC-IS)**, an ESA-funded, EO-driven digital twin project.

DTC-IS integrates Earth Observation, in-situ data, reanalysis products, and advanced data-driven models to provide a dynamic and science-based representation of the Greenland and Antarctic ice sheets, including their ice shelves. It supports the monitoring, understanding, and prediction of ice-sheet change, delivering actionable information for research, policy, and scenario-based exploration.

[Project Website](https://dtc-ice-sheets.org/) · [Interactive Dashboard](https://dashboards.dtc-ice-sheets.org/) · [Query API](https://query.dtc-ice-sheets.org)


## Climate Super Resolution

Global Climate Models (GCM) cover the whole planet but do not contain fine spatial details, 
whereas Regional Climate Models (RCMs) provide highly detailed information but cover small areas and come at 
a high additional computational cost. This module shows how machine learning can be used to emulate high resolution 
regional climate models having as input coarse global climate models, offering a cost efficient alternative to dynamical
downscaling.

Based on the work of Van der Meer et al. ([link to the paper](https://doi.org/10.1029/2022MS003593)), the model provided
here can predict the Surface Mass Balance (SMB) of Antarctica at high resolution, starting from eight selected variables of ERA5 monthly means.
The pretrained model is a UNet, that can be selected in its basic configuration, or with attention mechanism.

---

**Key features:**
- **Location:** Choose a point on the map, which will be the center of the analysed tile.
- **Analysis period:** Specify the start and end date within the ERA5 availability window (typically 1940–present).
- **Animation:** Visualise the predicted Surface Mass Balance (SMB) evolving over time.
- **Evaluate:** When ground truth data is present, check the model performance for a given prediction.

**How to use this notebook:**
1. Run the setup cells below.
2. **Select a point** on the map, around which a tile will be created.
3. Select a **time window**, for which to predict the SMB.
4. Explore the predictions over time, and their relation with the inputs and with the ground truth

**Requirements:**
- Jupyter (or VS Code with the Jupyter extension).
- Internet access to the DTC-IS API.
- A valid DTC_API_AUTH_TOKEN

---

In [ ]:
# query the api
import os
from datetime import datetime

import xarray as xr
from dtc_query_client import (
    ApiClient,
    SuperResolutionApi,
    SuperResolutionRcmEmulatorLocationRequest,
)
from dtc_query_client.helpers import wait_for_job
from unified_dashboard.query_api import get_api_config

%load_ext autoreload
%autoreload 2

To be able to query the DTC-API you need to set `DTC_API_AUTH_TOKEN` in the environment (you can find the token in the Account page of the Dashbord - bottom center):

In [ ]:
os.environ["DTC_API_AUTH_TOKEN"] = "" # copy paste your token here

In [ ]:
# from dotenv import load_dotenv
# load_dotenv()

# if "DTC_API_AUTH_TOKEN" not in os.environ:
#     raise EnvironmentError("DTC_API_AUTH_TOKEN must be set in the environment - create a .env file with the key.")

In [ ]:
%matplotlib widget
from src.dtc_is_notebook_helpers.uc4_plotting_helpers import select_location

selected = select_location()

## Select time window for the analysis

In [ ]:
START_DATE = datetime(2021, 1, 1)
END_DATE = datetime(2021, 6, 15)
LAT, LON = selected["lat"], selected["lon"]

MODEL_NAME = "model_baseline_unet" # OR "model_attention_unet"

### Run the Super Resolution RCM emulator entrypoint

The entrypoint will retrieve the input data through the API, and perform the following operations:
- Create a tile around the selected point -> the size is coherent with the model constraints
- Run the UNet model, taking as input 8 ERA5 variables over the selected tile
- Return the input ERA5, the model's predictions, and the target RACMO data if present

In [ ]:
async with ApiClient(get_api_config()) as client:
    print("Starting job...")

    res = await SuperResolutionApi(client).submit_rcm_emulator_location_job(
    SuperResolutionRcmEmulatorLocationRequest(
            start_date=START_DATE,
            end_date=END_DATE,
            latitude=LAT,
            longitude=LON,
            model_name="model_baseline_unet" # or "model_attention_unet" for the attention-based model
        ))

    # This should return the job ID of the started job
    job_id = res.job_id

    print("Waiting for job to complete...")
    # This line hangs until the job is complete, returning the final result.
    result = await wait_for_job(client, job_id)

    print(f"Job completed with status {result.status}")
    print("Outputs:")
    print(result.outputs)

In [ ]:
input_era5_ds = xr.open_dataset(result.outputs["data-download"]["output_path"][0])
groud_truth_racmo_ds = xr.open_dataset(result.outputs["data-download"]["output_path"][1])
out_ds = xr.open_dataset(result.outputs["rcm-emulator"]["output_path"])

## SMB predictions Explorer

This interactive explorer helps you visualize the results.

On the first row it shows one of the 8 input variables (total precipitation), the emulator's output and the RACMO reference data if available.  
Below, the timeseries shows the average value across the tile for each timestep within the selected window.

In [ ]:
from src.dtc_is_notebook_helpers.uc4_plotting_helpers import build_explorer

build_explorer(out_ds, stride=1).show()

### Visualize the results by date

In [ ]:
DATE = datetime(2021, 3, 21)

In [ ]:
from src.dtc_is_notebook_helpers.uc4_plotting_helpers import plot_single_date

fig = plot_single_date(out_ds, DATE)
fig

### Validation - if corresponding RACMO is available

In [ ]:
DATE = datetime(2021, 3, 21)

In [ ]:
from src.dtc_is_notebook_helpers.uc4_plotting_helpers import plot_validation

validation_fig = plot_validation(out_ds, groud_truth_racmo_ds, DATE)
validation_fig